# Urbanização e flags territoriais

Monta duas tabelas municipais usadas pela ABT final:

- População urbana e rural do Censo 2022, além das participações de urbanização e ruralização. Fonte: agregado 9923 do IBGE, preparado a partir de **01.Dados_IBGE_v2.ipynb** em **Clusterizacao_Cidades**.
- Flags de capital, município defrontante com o mar e pertencimento a região metropolitana oficial ou relevante. Fonte: **01.Dados_IBGE_v3.ipynb** em **Clusterizacao_Cidades**.

Uma região metropolitana é relevante quando contém capital, o maior município tem pelo menos 500 mil habitantes ou a população total chega a 1 milhão. A população é a do Censo 2022, vinda do livro do IBGE.

Entradas locais que não são baixadas:

- **data/trusted/ibge/dim_municipio.parquet**
- **data/analytics/abt_municipios_ibge_basica.parquet** (ou o painel / o censo, como alternativa)
- **data/raw/ibge/Municipios_Defrontantes_com_o_Mar_2024.xlsx** ou **.xls**

As regiões metropolitanas são lidas de **data/raw/ibge/ibge_rm_cidades.csv** ou **.json**. Se o arquivo não existir, o notebook baixa a resposta da API de localidades do IBGE.


## Importações


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from unidecode import unidecode


/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configuração

In [2]:
def resolve_repo_root() -> Path:
    """Localiza a raiz do projeto que contém os books e o diretório local de dados."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "books").is_dir() and (candidate / "data").is_dir():
            return candidate
    return start


REPO_ROOT = resolve_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "ibge"
TRUSTED_DIR = DATA_DIR / "trusted" / "ibge"
ANALYTICS_DIR = DATA_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DIM_PATH = TRUSTED_DIR / "dim_municipio.parquet"
POPULATION_CANDIDATES = [
    ANALYTICS_DIR / "abt_municipios_ibge_basica.parquet",
    ANALYTICS_DIR / "abt_municipios_ibge_painel.parquet",
    TRUSTED_DIR / "populacao_censos_2010_2022_municipio.parquet",
]
COASTAL_CANDIDATES = [
    RAW_DIR / "Municipios_Defrontantes_com_o_Mar_2024.xlsx",
    RAW_DIR / "Municipios_Defrontantes_com_o_Mar_2024.xls",
]
RM_CANDIDATES = [
    RAW_DIR / "ibge_rm_cidades.json",
    RAW_DIR / "ibge_rm_cidades.csv",
    TRUSTED_DIR / "ibge_rm_cidades.csv",
]
RM_DOWNLOAD_PATH = RAW_DIR / "ibge_rm_cidades.json"
RM_API_URL = "https://servicodados.ibge.gov.br/api/v1/localidades/regioes-metropolitanas"

URBAN_PATH = TRUSTED_DIR / "urbanizacao_municipio_censo_2022.parquet"
TERRITORY_PATH = TRUSTED_DIR / "municipio_litoral_rm.parquet"

KEY = "codigo_ibge"
CENSUS_YEAR = "2022"
AGGREGATE_ID = "9923"
POLE_POPULATION_MIN = 500_000
RM_POPULATION_MIN = 1_000_000

STATE_CAPITALS = {
    "AC": ("1200401", "Rio Branco"),
    "AL": ("2704302", "Maceió"),
    "AP": ("1600303", "Macapá"),
    "AM": ("1302603", "Manaus"),
    "BA": ("2927408", "Salvador"),
    "CE": ("2304400", "Fortaleza"),
    "DF": ("5300108", "Brasília"),
    "ES": ("3205309", "Vitória"),
    "GO": ("5208707", "Goiânia"),
    "MA": ("2111300", "São Luís"),
    "MT": ("5103403", "Cuiabá"),
    "MS": ("5002704", "Campo Grande"),
    "MG": ("3106200", "Belo Horizonte"),
    "PA": ("1501402", "Belém"),
    "PB": ("2507507", "João Pessoa"),
    "PR": ("4106902", "Curitiba"),
    "PE": ("2611606", "Recife"),
    "PI": ("2211001", "Teresina"),
    "RJ": ("3304557", "Rio de Janeiro"),
    "RN": ("2408102", "Natal"),
    "RS": ("4314902", "Porto Alegre"),
    "RO": ("1100205", "Porto Velho"),
    "RR": ("1400100", "Boa Vista"),
    "SC": ("4205407", "Florianópolis"),
    "SP": ("3550308", "São Paulo"),
    "SE": ("2800308", "Aracaju"),
    "TO": ("1721000", "Palmas"),
}


def first_existing(paths: list[Path]) -> Path:
    """Devolve o primeiro caminho que existe ou interrompe listando todos os candidatos."""
    for path in paths:
        if path.is_file():
            return path
    listed = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Nenhum dos arquivos esperados existe:\n{listed}")


## População urbana e rural

Tabela 9923 do SIDRA, população residente por situação do domicílio no Censo 2022.


In [3]:
def fetch_aggregate_metadata(aggregate_id: str) -> dict:
    """Carrega os metadados do agregado do IBGE."""
    url = f"https://servicodados.ibge.gov.br/api/v3/agregados/{aggregate_id}/metadados"
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    data = response.json()
    if isinstance(data, list):
        return data[0]
    return data


def find_resident_population_variable(metadata: dict) -> str:
    """Devolve o identificador da variável de população residente, excluindo variáveis de percentual."""
    for variable in metadata.get("variaveis", []):
        name = unidecode(variable.get("nome", "")).lower()
        if "populacao residente" in name and "percentual" not in name:
            return str(variable["id"])
    raise ValueError("A variável de população residente não foi encontrada nos metadados do agregado.")


def find_household_situation(metadata: dict) -> tuple[str, dict]:
    """Devolve o identificador da classificação de situação do domicílio e os identificadores das categorias."""
    for classification in metadata.get("classificacoes", []):
        name = unidecode(classification.get("nome", "")).lower()
        if "situacao" in name and "domicilio" in name:
            categories = {
                unidecode(category.get("nome", "")).lower().strip(): str(category["id"])
                for category in classification.get("categorias", [])
            }
            return str(classification["id"]), categories
    raise ValueError("A classificação de situação do domicílio não foi encontrada nos metadados do agregado.")


def download_urban_rural_population_2022() -> pd.DataFrame:
    """Baixa a população residente total, urbana e rural por município."""
    metadata = fetch_aggregate_metadata(AGGREGATE_ID)
    variable_id = find_resident_population_variable(metadata)
    classification_id, categories = find_household_situation(metadata)

    total_id = categories.get("total")
    urban_id = categories.get("urbana")
    rural_id = categories.get("rural")
    if total_id is None or urban_id is None or rural_id is None:
        raise ValueError(
            f"Esperava as categorias total, urbana e rural. Encontradas: {categories}"
        )

    category_ids = ",".join([total_id, urban_id, rural_id])
    url = (
        f"https://servicodados.ibge.gov.br/api/v3/agregados/{AGGREGATE_ID}"
        f"/periodos/{CENSUS_YEAR}/variaveis/{variable_id}"
        f"?localidades=N6[all]&classificacao={classification_id}[{category_ids}]"
    )
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    payload = response.json()

    records = []
    for result in payload[0]["resultados"]:
        classifications = result.get("classificacoes", [])
        situation_name = None
        if classifications:
            category = classifications[0].get("categoria", {})
            if category:
                situation_name = list(category.values())[0]
        for series in result.get("series", []):
            locality = series.get("localidade", {})
            records.append({
                KEY: str(locality.get("id", "")).zfill(7),
                "municipio_sidra": locality.get("nome"),
                "situacao_domicilio": situation_name,
                "populacao": pd.to_numeric(series.get("serie", {}).get(CENSUS_YEAR), errors="coerce"),
            })

    population = pd.DataFrame(records)
    population["situacao_domicilio_norm"] = (
        population["situacao_domicilio"]
        .astype("string")
        .map(lambda value: unidecode(str(value)).lower().strip())
    )
    wide = (
        population.pivot_table(
            index=[KEY, "municipio_sidra"],
            columns="situacao_domicilio_norm",
            values="populacao",
            aggfunc="sum",
        )
        .reset_index()
        .rename(columns={
            "total": "populacao_total_censo_2022",
            "urbana": "populacao_urbana_2022",
            "rural": "populacao_rural_2022",
        })
    )
    for column in [
        "populacao_total_censo_2022",
        "populacao_urbana_2022",
        "populacao_rural_2022",
    ]:
        if column not in wide.columns:
            wide[column] = np.nan

    total = wide["populacao_total_censo_2022"].replace(0, np.nan)
    wide["indice_urbanizacao_2022"] = wide["populacao_urbana_2022"] / total
    wide["indice_ruralizacao_2022"] = wide["populacao_rural_2022"] / total
    return wide.sort_values(KEY).reset_index(drop=True)


urbanizacao_2022 = download_urban_rural_population_2022()
urbanizacao_2022.to_parquet(URBAN_PATH, index=False)
print(f"Salvo em {URBAN_PATH} {urbanizacao_2022.shape}")
urbanizacao_2022.head()


Saved /Users/lucas.nascimento/city-information-books/data/trusted/ibge/urbanizacao_municipio_censo_2022.parquet (5570, 7)


situacao_domicilio_norm,codigo_ibge,municipio_sidra,populacao_rural_2022,populacao_total_censo_2022,populacao_urbana_2022,indice_urbanizacao_2022,indice_ruralizacao_2022
0,1100015,Alta Floresta D'Oeste - RO,8523.0,21494.0,12971.0,0.603471,0.396529
1,1100023,Ariquemes - RO,12881.0,96833.0,83952.0,0.866977,0.133023
2,1100031,Cabixi - RO,2505.0,5351.0,2846.0,0.531863,0.468137
3,1100049,Cacoal - RO,14980.0,86887.0,71907.0,0.827592,0.172408
4,1100056,Cerejeiras - RO,1774.0,15890.0,14116.0,0.888357,0.111643


## Capital e litoral

In [4]:
def normalize_column_name(column: str) -> str:
    """Transforma um cabeçalho de origem em um nome no formato snake_case."""
    normalized = unidecode(str(column).strip()).lower()
    normalized = re.sub(r"[^a-z0-9]+", "_", normalized)
    return re.sub(r"_+", "_", normalized).strip("_")


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [normalize_column_name(column) for column in out.columns]
    return out


def normalize_codigo_ibge(series: pd.Series) -> pd.Series:
    """Converte o código do município para uma chave de texto com 7 dígitos."""
    values = series.astype("string").str.replace(r"\D", "", regex=True)
    return values.str.zfill(7)


def state_capital_table() -> pd.DataFrame:
    """Uma linha por estado com o código oficial do município da capital."""
    rows = [
        {"uf": state, "codigo_capital_uf": code, "nome_capital_uf": name}
        for state, (code, name) in STATE_CAPITALS.items()
    ]
    capitals = pd.DataFrame(rows)
    capitals["codigo_capital_uf"] = normalize_codigo_ibge(capitals["codigo_capital_uf"])
    return capitals


def add_capital_flag(municipalities: pd.DataFrame) -> pd.DataFrame:
    """Marca o município que é a capital do seu estado."""
    out = municipalities.copy()
    out[KEY] = normalize_codigo_ibge(out[KEY])
    out["uf"] = out["uf"].astype("string").str.upper().str.strip()
    out = out.merge(state_capital_table(), on="uf", how="left")
    out["flag_capital_uf"] = (out[KEY] == out["codigo_capital_uf"]).fillna(False).astype(np.int64)
    return out


def read_coastal_file(path: Path) -> pd.DataFrame:
    """Lê a lista do IBGE de municípios defrontantes com o mar."""
    suffix = path.suffix.lower()
    if suffix == ".parquet":
        frame = pd.read_parquet(path)
    elif suffix in {".xlsx", ".xls", ".ods"}:
        frame = pd.read_excel(path, dtype="string")
    elif suffix == ".csv":
        frame = pd.read_csv(path, dtype="string", encoding="utf-8")
        if frame.shape[1] == 1:
            frame = pd.read_csv(path, sep=";", dtype="string", encoding="utf-8")
    else:
        raise ValueError(f"Extensão não suportada no arquivo de litoral: {suffix}")
    return normalize_columns(frame)


def standardize_coastal_municipalities(coastal: pd.DataFrame) -> pd.DataFrame:
    """Mantém o código de 7 dígitos, a alternativa de 6 dígitos e a flag de litoral."""
    code_candidates = [
        "codigo_ibge",
        "cod_ibge",
        "geocodigo",
        "geocodig_m",
        "cd_mun",
        "cd_municipio",
        "cod_mun",
        "codigo_municipio",
        "municipio",
    ]
    code_column = None
    for column in code_candidates:
        if column not in coastal.columns:
            continue
        lengths = coastal[column].astype("string").str.replace(r"\D", "", regex=True).str.len()
        if lengths.isin([6, 7]).mean() > 0.5:
            code_column = column
            break
    if code_column is None:
        raise ValueError(
            "Nenhuma coluna de código municipal foi encontrada no arquivo de litoral. "
            f"Colunas: {coastal.columns.tolist()}"
        )

    code = coastal[code_column].astype("string").str.replace(r"\D", "", regex=True)
    out = pd.DataFrame({
        "codigo_ibge_6": code.str.zfill(6).str[:6],
        KEY: np.where(code.str.len() == 7, code.str.zfill(7), pd.NA),
        "flag_litoranea_ibge": 1,
    })
    return out.drop_duplicates([KEY, "codigo_ibge_6"])


def add_coastal_flag(municipalities: pd.DataFrame, coastal_path: Path) -> pd.DataFrame:
    """Marca os municípios defrontantes com o mar. Os que não casam permanecem 0."""
    out = municipalities.copy()
    out[KEY] = normalize_codigo_ibge(out[KEY])
    out["codigo_ibge_6"] = out[KEY].str[:6]
    coastal = standardize_coastal_municipalities(read_coastal_file(coastal_path))

    coastal_7 = coastal[[KEY, "flag_litoranea_ibge"]].dropna().drop_duplicates(KEY)
    out = out.merge(coastal_7, on=KEY, how="left")
    if out["flag_litoranea_ibge"].isna().any():
        coastal_6 = (
            coastal[["codigo_ibge_6", "flag_litoranea_ibge"]]
            .dropna()
            .drop_duplicates("codigo_ibge_6")
            .rename(columns={"flag_litoranea_ibge": "flag_litoranea_ibge_6"})
        )
        out = out.merge(coastal_6, on="codigo_ibge_6", how="left")
        out["flag_litoranea_ibge"] = out["flag_litoranea_ibge"].fillna(out["flag_litoranea_ibge_6"])
        out = out.drop(columns=["flag_litoranea_ibge_6"])

    out["flag_litoranea_ibge"] = (
        pd.to_numeric(out["flag_litoranea_ibge"], errors="coerce").fillna(0).astype(np.int64)
    )
    return out


## Região metropolitana

In [5]:
def load_metropolitan_regions(path: Path) -> list:
    """Carrega o arquivo de regiões metropolitanas do IBGE. Se o arquivo não existir, ele é baixado."""
    if not path.is_file():
        response = requests.get(RM_API_URL, timeout=120)
        response.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(response.text, encoding="utf-8")

    text = path.read_text(encoding="utf-8-sig").strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        data = pd.read_json(path, encoding="utf-8-sig").to_dict("records")

    if isinstance(data, dict):
        return [data]
    if isinstance(data, list):
        return data
    raise ValueError(f"Esperava uma lista JSON de regiões metropolitanas e recebeu {type(data).__name__}.")


def flatten_metropolitan_regions(regions: list) -> pd.DataFrame:
    """Expande o arquivo aninhado do IBGE para uma linha por município e região."""
    rows = []
    for region in regions:
        state = region.get("UF", {}) or {}
        macro_region = state.get("regiao", {}) or {}
        for municipality in region.get("municipios", []) or []:
            rows.append({
                KEY: str(municipality.get("id")).zfill(7),
                "municipio_rm_api": municipality.get("nome"),
                "id_regiao_metropolitana": str(region.get("id")).zfill(5),
                "nome_regiao_metropolitana": region.get("nome"),
                "codigo_uf_rm": state.get("id"),
                "uf_rm": state.get("sigla"),
                "nome_uf_rm": state.get("nome"),
                "codigo_regiao_rm": macro_region.get("id"),
                "sigla_regiao_rm": macro_region.get("sigla"),
                "regiao_rm": macro_region.get("nome"),
                "flag_regiao_metropolitana_ibge": 1,
            })
    if not rows:
        raise ValueError("O arquivo de regiões metropolitanas não tem municípios.")

    out = pd.DataFrame(rows)
    out[KEY] = normalize_codigo_ibge(out[KEY])
    out["id_regiao_metropolitana"] = (
        out["id_regiao_metropolitana"].astype("string").str.replace(r"\D", "", regex=True).str.zfill(5)
    )
    out["uf_rm"] = out["uf_rm"].astype("string").str.upper().str.strip()
    return (
        out.drop_duplicates([KEY, "id_regiao_metropolitana"])
        .sort_values(["uf_rm", "nome_regiao_metropolitana", "municipio_rm_api"])
        .reset_index(drop=True)
    )


def prepare_population_for_regions(municipalities: pd.DataFrame) -> pd.DataFrame:
    """Mantém uma população do Censo 2022 por município para as regras de região."""
    out = municipalities.copy()
    out[KEY] = normalize_codigo_ibge(out[KEY])
    if "populacao_censo_2022" not in out.columns:
        raise ValueError("A coluna populacao_censo_2022 era esperada na tabela municipal do IBGE.")
    columns = [KEY, "populacao_censo_2022"]
    for column in ["municipio", "uf"]:
        if column in out.columns:
            columns.append(column)
    out = out[columns].rename(columns={"populacao_censo_2022": "populacao_municipio"})
    out["populacao_municipio"] = pd.to_numeric(out["populacao_municipio"], errors="coerce")
    return out.dropna(subset=[KEY]).drop_duplicates(KEY, keep="last").reset_index(drop=True)


def classify_relevant_regions(
    region_municipalities: pd.DataFrame,
    population: pd.DataFrame,
) -> pd.DataFrame:
    """Anexa as flags de relevância. Um município pode pertencer a mais de uma região."""
    regions = region_municipalities.copy()
    regions[KEY] = normalize_codigo_ibge(regions[KEY])
    population = population.copy()
    population[KEY] = normalize_codigo_ibge(population[KEY])

    classified = regions.merge(population, on=KEY, how="left")
    if "municipio" not in classified.columns:
        classified["municipio"] = classified["municipio_rm_api"]
    if "uf" not in classified.columns:
        classified["uf"] = classified["uf_rm"]

    capitals = state_capital_table()[["codigo_capital_uf"]]
    classified = classified.merge(
        capitals,
        left_on=KEY,
        right_on="codigo_capital_uf",
        how="left",
    )
    classified["flag_municipio_capital"] = classified["codigo_capital_uf"].notna().astype(int)
    classified["populacao_municipio"] = pd.to_numeric(
        classified["populacao_municipio"], errors="coerce"
    )

    summary = classified.groupby(
        [
            "id_regiao_metropolitana",
            "nome_regiao_metropolitana",
            "uf_rm",
            "nome_uf_rm",
            "regiao_rm",
        ],
        as_index=False,
    ).agg(
        qtd_municipios_rm=(KEY, "nunique"),
        populacao_total_rm=("populacao_municipio", "sum"),
        populacao_municipio_polo_rm=("populacao_municipio", "max"),
        flag_rm_contem_capital=("flag_municipio_capital", "max"),
    )
    summary["flag_rm_polo_grande"] = (
        summary["populacao_municipio_polo_rm"] >= POLE_POPULATION_MIN
    ).astype(int)
    summary["flag_rm_pop_total_grande"] = (
        summary["populacao_total_rm"] >= RM_POPULATION_MIN
    ).astype(int)
    summary["flag_regiao_metropolitana_relevante"] = (
        summary["flag_rm_contem_capital"].eq(1)
        | summary["flag_rm_polo_grande"].eq(1)
        | summary["flag_rm_pop_total_grande"].eq(1)
    ).astype(int)

    return classified.merge(
        summary[
            [
                "id_regiao_metropolitana",
                "qtd_municipios_rm",
                "populacao_total_rm",
                "flag_rm_contem_capital",
                "flag_rm_polo_grande",
                "flag_rm_pop_total_grande",
                "flag_regiao_metropolitana_relevante",
            ]
        ],
        on="id_regiao_metropolitana",
        how="left",
    )


def consolidate_regions_by_municipality(classified: pd.DataFrame) -> pd.DataFrame:
    """Consolida várias regiões em uma linha por município."""
    out = classified.groupby(KEY, as_index=False).agg(
        flag_regiao_metropolitana_ibge=("flag_regiao_metropolitana_ibge", "max"),
        flag_regiao_metropolitana_relevante=("flag_regiao_metropolitana_relevante", "max"),
        qtd_municipios_rm=("qtd_municipios_rm", "max"),
        populacao_total_rm=("populacao_total_rm", "max"),
        flag_rm_contem_capital=("flag_rm_contem_capital", "max"),
        flag_rm_polo_grande=("flag_rm_polo_grande", "max"),
        flag_rm_pop_total_grande=("flag_rm_pop_total_grande", "max"),
    )
    for column in [
        "flag_regiao_metropolitana_ibge",
        "flag_regiao_metropolitana_relevante",
        "flag_rm_contem_capital",
        "flag_rm_polo_grande",
        "flag_rm_pop_total_grande",
    ]:
        out[column] = out[column].fillna(0).astype(int)
    return out


## Base territorial

Municípios fora de qualquer região metropolitana ficam com zero nas flags, nas quantidades e na população da região.


In [6]:
if not DIM_PATH.is_file():
    raise FileNotFoundError(f"Arquivo ausente: {DIM_PATH}. Execute antes 01_dados_ibge.ipynb.")

dim_municipio = pd.read_parquet(DIM_PATH)
dim_municipio[KEY] = normalize_codigo_ibge(dim_municipio[KEY])
dim_municipio = dim_municipio.drop_duplicates(KEY)

coastal_path = first_existing(COASTAL_CANDIDATES)
dim_territorio = add_capital_flag(dim_municipio)
dim_territorio = add_coastal_flag(dim_territorio, coastal_path)

rm_path = next((path for path in RM_CANDIDATES if path.is_file()), RM_DOWNLOAD_PATH)
metropolitan_regions = flatten_metropolitan_regions(load_metropolitan_regions(rm_path))
population = prepare_population_for_regions(
    pd.read_parquet(first_existing(POPULATION_CANDIDATES))
)
regions_by_municipality = consolidate_regions_by_municipality(
    classify_relevant_regions(metropolitan_regions, population)
)

territory_flags = dim_territorio[[KEY, "flag_litoranea_ibge", "flag_capital_uf"]]
municipio_litoral_rm = territory_flags.merge(
    regions_by_municipality,
    on=KEY,
    how="left",
).fillna(0)
municipio_litoral_rm = municipio_litoral_rm.drop_duplicates(KEY).sort_values(KEY).reset_index(drop=True)
municipio_litoral_rm.to_parquet(TERRITORY_PATH, index=False)

print(f"Salvo em {TERRITORY_PATH} {municipio_litoral_rm.shape}")
print(municipio_litoral_rm[
    [
        "flag_litoranea_ibge",
        "flag_capital_uf",
        "flag_regiao_metropolitana_ibge",
        "flag_regiao_metropolitana_relevante",
    ]
].sum())
municipio_litoral_rm.head()


Saved /Users/lucas.nascimento/city-information-books/data/trusted/ibge/municipio_litoral_rm.parquet (5571, 10)
flag_litoranea_ibge                     279.0
flag_capital_uf                          27.0
flag_regiao_metropolitana_ibge         1377.0
flag_regiao_metropolitana_relevante     554.0
dtype: float64


,codigo_ibge,flag_litoranea_ibge,flag_capital_uf,flag_regiao_metropolitana_ibge,flag_regiao_metropolitana_relevante,qtd_municipios_rm,populacao_total_rm,flag_rm_contem_capital,flag_rm_polo_grande,flag_rm_pop_total_grande
0,1100015,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1100023,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1100031,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1100049,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1100056,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
